> 可运行笔记：每个 ```python 块都是一个能直接跑的 cell。
> V2 那个 cell 会报 `TypeError`——**那是故意的**，它就是自测里的「卡住」现场。

# d01 · 2026-10-25（周日）· 建词汇表 + 拿报文直觉

> 全天约 2.5 小时。顺序不能换：先自测，再读。
> 交付就是本文件填完 + `PROGRESS.md` 加一行变更日志。

---

## 0. 门槛自测（30 min，**关掉 AI、不查资料**）

任务：写一个 40 行的「假 agent」

1. 一个 dict 里放两个函数：`get_weather`、`control_ac`
2. 输入一条指令（例如 `"把空调调到24度"`）
3. `for` 循环最多跑 3 步
4. 每步挑一个函数调用并打印结果
5. `try/except` 接住函数抛的错，打印后**继续**循环

### 我写的代码

#### V1

In [ ]:
temperature=27
def run(command:str ,max_steps:int=3)->None:
    if command.find("空调")==-1:
        print("未知命令")
        return
    digit="".join(ch for ch in command if ch.isdigit())
    try:
        temp=int(digit)
    except ValueError:
        print("输入参数没有对应温度")
        return
    if 16<=temp<=30:
        temperature=temp
        print(f"已将温度修改为{temp}")
    else:
        print("输入温度错误")

if __name__ == '__main__':
    run("今天吃什么？")
    run("把空调温度调到-273度。")
    run("把空调温度调到16度。")

#### V2

In [ ]:

def get_weather():pass
def control_ac():pass
TOOL_REGISTRY={"get_weather":get_weather,"control_ac":control_ac}
def invoke(prompt:str|dict):pass
def run(command:str ,max_steps:int=3)->None:
    for step in range(1,max_steps+1):
        try:
            fun_name,argvs=invoke(command)
        except OSError as err:
            print("OS Error: ",err)
        TOOL_REGISTRY[fun_name](argvs)

if __name__ == '__main__':
    run("今天吃什么？")
    run("把空调温度调到-273度。")
    run("把空调温度调到16度。")
    run("把空调调到-20度")

### 自评

- [ ] 独立写完了
- [1] 卡住了，卡在：`将command中的温度参数提取出来`
- 写完后我觉得最别扭的一行是：`在将command参数的温度提取的同时没有提取正负号，并且如果出现多个数字这个程序会有bug`

- [2]自评2：`按照agent的思路写出了第二个版本，fun函数实现交给agent编写，只写了agent的大体框架`
> 结论决定了 W1-A 是 2 天还是 3 天，别美化。

#### V3｜正确结构对照（10/30 补：跑通之后才知道前两版差在哪）

| V2 的问题 | V3 的处理 | 落在哪一行 |
|---|---|---|
| `invoke()` 返回 `None`，一解包就 `TypeError` | `fake_model()` 返回 `{"tool_calls":[...]}` 或 `{"content":"..."}`，形状与真响应一致 | `run()` 第一行 |
| `except OSError` 抓不住 `KeyError` / `TypeError` / 业务错 | 业务错由工具 `return {"ok":False,...}` 掉，异常不穿过循环；JSON 解析失败单独接 | 工具 + 内层 `try` |
| 循环没有停止条件，跑满 3 步空转 | 停止条件 = 响应里**没有** `tool_calls` | `if not tool_calls: return` |
| 工具全是 `pass`，没有真对象 | 工具只改 `state` 的键，终态从 `state` 读 | `state["ac_temp"] = temp` |
| `-16` 被抽成 `16`，静默落在合法区间 | 参数进来先验类型、再验范围，校验放在**工具**里而不是抽取处 | `control_ac()` 两个 `if` |

In [ ]:
# d01 对照版：40 行的"假 agent"，结构 1:1 对齐真 agent
# WHY: 这里没有网络、没有模型。fake_model() 顶替的只是"模型返回一段响应"这一步，
#      其余每一环（拼上下文 / 解析 tool_calls / 执行 / 回灌 / 判停）和你接真模型时一模一样。
#      结构错了，接上真模型只会错得更快。

import json
import re

# WHY: 状态放独立容器，工具只改它的键，run() 只读它。
# TRAP: 写成 state = {...} 再在函数里 ac_temp = temp，那是一次局部重新绑定，
#       全局那个不会变——V1 就是这么"报成功"的。
state = {"ac_temp": 27, "ac_on": False}


def control_ac(args: dict) -> dict:
    # WHY: 参数一律按 dict 进来。真模型的 tool_call.arguments 就是一段 JSON 字符串，
    #      json.loads 之后正好是 dict，所以这里的形状和真的一模一样。
    temp = args.get("temperature")
    if not isinstance(temp, int) or isinstance(temp, bool):
        return {"ok": False, "error": f"temperature 必须是整数，收到 {temp!r}",
                "hint": "例如 {\"temperature\": 24}"}
    if not 16 <= temp <= 30:
        return {"ok": False, "error": f"温度 {temp} 超出 16~30",
                "hint": "请给 16 到 30 之间的整数"}      # TRAP: V1 缺的就是这条。
    state["ac_temp"] = temp                            # 改键，不重新绑定名字
    state["ac_on"] = True
    return {"ok": True, "data": {"ac_temp": temp, "ac_on": True}}


def get_weather(args: dict) -> dict:
    city = args.get("city")
    if not city:
        return {"ok": False, "error": "缺少参数 city", "hint": "例如 {\"city\": \"重庆\"}"}
    return {"ok": True, "data": {"city": city, "text": "多云 22℃"}}


# YOU: 第三个工具自己写（比如 set_reminder），契约同上：dict 进、{ok,...} 出、只改 state 的键。
TOOLS = {"control_ac": control_ac, "get_weather": get_weather}


def fake_model(messages: list) -> dict:
    # WHY: 用它顶替 LLM。行为固定 → 可复现。数字从指令里正则抠出来只是替身的权宜之计，
    #      不是你将来要写的逻辑：真实场景里这步是模型做的。
    # YOU: 接真模型时删掉整个函数，换成 client.chat(messages, tools=TOOL_SCHEMAS)。
    if any(m["role"] == "tool" for m in messages):
        # TRAP: 这里故意无视工具结果、直接宣布成功——复刻真实模型最常见的故障：
        #       工具明明失败了它照样说"已完成"。所以终态断言必须读 state，不能读这句话。
        num = re.search(r"-?\d+", messages[0]["content"])
        return {"content": f"已把空调调到 {num.group()} 度。"}
    num = re.search(r"-?\d+", messages[0]["content"])
    args = {"temperature": int(num.group())} if num else {}
    return {"tool_calls": [{"id": "call_1",
                            "function": {"name": "control_ac",
                                         "arguments": json.dumps(args)}}]}


def run(instruction: str, max_steps: int = 3) -> str:
    messages = [{"role": "user", "content": instruction}]
    for step in range(1, max_steps + 1):
        resp = fake_model(messages)
        tool_calls = resp.get("tool_calls")
        if not tool_calls:
            return resp.get("content", "")          # WHY: 唯一正常的终止条件
        messages.append({"role": "assistant", "tool_calls": tool_calls})
        for tc in tool_calls:
            name = tc["function"]["name"]
            try:
                args = json.loads(tc["function"]["arguments"])   # TRAP: arguments 是字符串
            except json.JSONDecodeError:
                result = {"ok": False, "error": "arguments 不是合法 JSON",
                          "hint": "常见于模型输出被截断"}
            else:
                fn = TOOLS.get(name)
                if fn is None:
                    result = {"ok": False, "error": f"没有工具 {name}",
                              "hint": f"可用：{list(TOOLS)}"}
                else:
                    result = fn(args)               # WHY: 业务错由工具自己 return 掉，
                    if not isinstance(result, dict):  # 不用异常穿过循环
                        result = {"ok": False, "error": "工具没返回 dict", "hint": "统一 {ok,...}"}
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "content": json.dumps(result, ensure_ascii=False)})
    return "达到最大步数，未完成"                    # YOU: 改成"回灌一条超时消息给模型再决定"


if __name__ == "__main__":
    for ins in ["把空调调到 24 度", "把空调调到 -16 度", "把空调调到 99 度"]:
        print("指令:", ins)
        print("  模型说:", run(ins))
        print("  真实状态:", state)                  # WHY: 终态读这里，不读上面那句话

跑出来的真实输出（Python 3.15，`python3.15 d01_v3.py`）：

```
指令: 把空调调到 24 度
  模型说: 已把空调调到 24 度。
  真实状态: {'ac_temp': 24, 'ac_on': True}
指令: 把空调调到 -16 度
  模型说: 已把空调调到 -16 度。
  真实状态: {'ac_temp': 24, 'ac_on': True}
指令: 把空调调到 99 度
  模型说: 已把空调调到 99 度。
  真实状态: {'ac_temp': 24, 'ac_on': True}
```

后两条就是 AGENTS.md §7 那条规矩的现场演示：模型说"已完成"，`state` 里根本没动。终态断言只能读 `state`，读那句话就是在测模型的嘴。

**`# YOU:` 三处——删掉所有注释之后，这三处还得写得出来：**

1. 第三个工具（比如 `set_reminder`），契约同前：dict 进、`{ok,...}` 出、只改 `state` 的键
2. `fake_model()` 换成真 `client.chat(messages, tools=...)`，里面那行 `json.loads(arguments)` 必须保留（`arguments` 永远是字符串）
3. `max_steps` 用尽时别直接 `return` 一个字符串——回灌一条"超时"消息，让模型自己决定收尾还是放弃

> **这份是对照物，不是交付。** 用法：不看它，自己把 `src/agent/loop.py` 写完，再回来逐行 diff，对不上的地方就是认知缺口。现在照抄一遍，等于把缺口盖住。

---

## 1. A4｜七个子系统（20 min）

链接：https://arxiv.org/abs/2609.00006
只读：七个子系统的**名字 + 一句话职责**，以及结论。**跳过最小/最大实现对比。**

| #   | 子系统     | 一句话职责（自己的话） | 我的作品里对应哪个文件 |
| --- | ---------- | ---------------------- | ---------------------- |
| 1   | 主循环     |  推理和提示交替运行的循环 | `src/agent/loop.py`    |
| 2   | 工具系统   | 定义agent可以做的是 | `src/tools/`           |
| 3   | 上下文管理 | 管理会话中的历史 | `src/context/`         |
| 4   | 记忆       | agent在会话中总结的知识 | `src/memory/`          |
| 5   | 安全与权限 | 确定agent什么请求可以执行什么不可以 | `src/safety/`          |
| 6   | 编排       | 协调多agent工作 | 暂不做                 |
| 7   | 扩展面     | 插件，skill | 暂不做                 |

**结论那节里，我记住的两条：**

1. harness和langchain框架的区别是：langchain是import进来搭agent的库,而harness则是在agent内部工作运行时
2. Aider故意缺席了Orchestration（编排）部分，因为Aider将资源压在了编辑精度上，用单agent换取了责任可追溯。

---

## 2. 报文样例（60 min）

链接：你所用模型的 Tool Use / Function Calling 文档
（参考：https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview）

### 请求报文（脱敏，去掉 key）

```json

```

### 响应报文（脱敏）

```json

```

### 我从报文里看出的三个细节

1. `tool_calls` 里最关键的字段是：`______`，因为 `______`
2. 工具结果回灌时，用的是什么角色 / 什么字段：`______`
3. 一轮要调多个工具时，区别是：`______`

### 我还没看懂的地方

- ***

## 3. 工具描述五条自查（30 min）

链接：https://www.anthropic.com/engineering/writing-tools-for-agents
只读：「What is a tool」+「Prompt-engineering your tool descriptions」

我抄下来的自查项（写成能打勾的句子）：

- [ ]
- [ ]
- [ ]
- [ ]
- [ ]

---

## 4. 今天踩的坑 / 意外发现

- ***

## 5. 明天（d02）开工前，我要先改的一件事

-